In [1]:
### Imports ###

import requests
import math
import os

In [2]:
# Dictionary with the latitude, longitude, and the
# window width and height (in km) of each of the 9 
# reservoirs in our dataset.  

catalan_dams = {
    "riudecanyes": {
        "lat": 41.1354, 
        "lon": 0.9534,
        "width_km": 2,
        "height_km": 2
    },
    "siurana": {
        "lat": 41.2546, 
        "lon": 0.9198,
        "width_km": 3,
        "height_km": 2
    },
    "llosa de cavall": {
        "lat": 42.1199, 
        "lon": 1.6016,
        "width_km": 7,
        "height_km": 6
    },
    "sant ponç": {
        "lat": 41.9794, 
        "lon": 1.5933,
        "width_km": 5,
        "height_km": 5
    },
    "susqueda": {
        "lat": 41.9714,  
        "lon": 2.4953,
        "width_km": 10,
        "height_km": 10
    },
    "boadella": {
        "lat": 42.3491,  
        "lon": 2.8133,
        "width_km": 5,
        "height_km": 5
    },
    "sau": {
        "lat": 41.9775,
        "lon": 2.3702,
        "width_km": 8, 
        "height_km": 8
    },
    "baells": {
        "lat": 42.1472,
        "lon": 1.8830,
        "width_km": 6,
        "height_km": 8
    },
    "foix": {
        "lat": 41.2582,
        "lon": 1.6401,
        "width_km": 2,
        "height_km": 2
    }
}

In [ ]:
# For each of the reservoirs, we download a recent
# RGB satellite image to verify that the window
# specified in in the dictionary is optimal. 

# Authentication

with open('../confidential/SH_CLIENT_ID.txt', 'r') as file:
    CLIENT_ID = file.read()

with open('../confidential/SH_CLIENT_SECRET.txt', 'r') as file:
    CLIENT_SECRET = file.read()

auth_url = 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token'
response = requests.post(auth_url, data={
    'grant_type': 'client_credentials', 'client_id': CLIENT_ID, 'client_secret': CLIENT_SECRET
})
token = response.json().get('access_token')

# Helper function that computes the window from
# the specifications in the reservoir dictionary.
# By diving by 111 (the approximate distance in 
# kilometers between two degrees of latitude), we
# transform  km in degrees

def get_custom_bbox(lat, lon, width_km, height_km):
    lat_change = (height_km / 2) / 111.0
    lon_change = (width_km / 2) / (111.0 * math.cos(math.radians(lat)))
    return [lon - lon_change, lat - lat_change, lon + lon_change, lat + lat_change]

# Pull a RGB image and brightens them slightly

evalscript_rgb = """
//VERSION=3
function setup() {
  return {
    input: ["B04", "B03", "B02", "dataMask"],
    output: { bands: 3 }
  };
}
function evaluatePixel(sample) {
  if (sample.dataMask === 0) return [0, 0, 0]; // Black for no data
  return [2.5 * sample.B04, 2.5 * sample.B03, 2.5 * sample.B02]; 
}
"""

process_url = 'https://sh.dataspace.copernicus.eu/api/v1/process'
headers = {
    'Authorization': f'Bearer {token}',
    'Accept': 'image/jpeg'
}

# The images to visually verify the window are
# to be stored in the directory data/window_checks 

os.makedirs("../data/window_checks", exist_ok=True)

for dam_name, info in catalan_dams.items():
    print(f"Fetching recent RGB image for {dam_name}...")
    
    bbox = get_custom_bbox(info['lat'], info['lon'], info['width_km'], info['height_km'])
    
    # Calculate pixels to maintain exactly 20 meters per pixel, preserving aspect ratio
    pixels_x = int((info["width_km"] * 1000) / 20)
    pixels_y = int((info["height_km"] * 1000) / 20)
    
    payload = {
        "input": {
            "bounds": {"bbox": bbox},
            "data": [{
                "dataFilter": {
                    # Looking back 30 days from today (March 2026) to find a clear day
                    "timeRange": {"from": "2026-01-05T00:00:00Z", "to": "2026-03-07T23:59:59Z"},
                    "maxCloudCoverage": 15 # Only accept mostly clear days
                },
                "type": "sentinel-2-l2a"
            }]
        },
        "output": {
            "width": pixels_x, 
            "height": pixels_y,
            "responses": [{"identifier": "default", "format": {"type": "image/jpeg"}}]
        },
        "evalscript": evalscript_rgb
    }
    
    res = requests.post(process_url, headers=headers, json=payload)
    
    if res.status_code == 200:
        filepath = f"../data/window_checks/{dam_name}_rgb_check.jpg"
        with open(filepath, 'wb') as f:
            f.write(res.content)
        print(f" -> Saved {filepath} ({pixels_x}x{pixels_y} pixels)")
    else:
        print(f" -> Failed for {dam_name}: {res.status_code} - {res.text}")

print("All done! Go check the 'window_checks' folder.")

Fetching recent RGB image for riudecanyes...
 -> Saved dam_checks/riudecanyes_rgb_check.jpg (100x100 pixels)
Fetching recent RGB image for siurana...
 -> Saved dam_checks/siurana_rgb_check.jpg (150x100 pixels)
Fetching recent RGB image for llosa de cavall...
 -> Saved dam_checks/llosa de cavall_rgb_check.jpg (350x300 pixels)
Fetching recent RGB image for sant ponç...
 -> Saved dam_checks/sant ponç_rgb_check.jpg (250x250 pixels)
Fetching recent RGB image for susqueda...
 -> Saved dam_checks/susqueda_rgb_check.jpg (500x500 pixels)
Fetching recent RGB image for boadella...
 -> Saved dam_checks/boadella_rgb_check.jpg (250x250 pixels)
Fetching recent RGB image for sau...
 -> Saved dam_checks/sau_rgb_check.jpg (400x400 pixels)
Fetching recent RGB image for baells...
 -> Saved dam_checks/baells_rgb_check.jpg (300x400 pixels)
Fetching recent RGB image for foix...
 -> Saved dam_checks/foix_rgb_check.jpg (100x100 pixels)
All done! Go check the 'dam_checks' folder.


In [4]:
# Once we have verified the windows, we compute the
# NDWI in the server and download the results for
# all clear days (cloud_cover <= 15) from January 
# 2020 to March 2026. 

# Authentication

with open('../confidential/SH_CLIENT_ID.txt', 'r') as file:
    CLIENT_ID = file.read()

with open('../confidential/SH_CLIENT_SECRET.txt', 'r') as file:
    CLIENT_SECRET = file.read()

auth_url = 'https://identity.dataspace.copernicus.eu/auth/realms/CDSE/protocol/openid-connect/token'
res = requests.post(auth_url, data={'grant_type': 'client_credentials', 'client_id': CLIENT_ID, 'client_secret': CLIENT_SECRET})
token = res.json().get('access_token')

headers = {'Authorization': f'Bearer {token}', 'Accept': 'image/tiff'}
catalog_headers = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}

# Helper function that computes the window from
# the specifications in the reservoir dictionary.
# By diving by 111 (the approximate distance in 
# kilometers between two degrees of latitude), we
# transform  km in degrees

def get_custom_bbox(lat, lon, width_km, height_km):
    lat_change = (height_km / 2) / 111.0
    lon_change = (width_km / 2) / (111.0 * math.cos(math.radians(lat)))
    return [lon - lon_change, lat - lat_change, lon + lon_change, lat + lat_change]

# Script that computes the Normalized Difference
# Water Index (NDWI). 

evalscript_ndwi = """
//VERSION=3
function setup() {
  return {
    input: ["B03", "B08", "dataMask"],
    output: { bands: 1, sampleType: "FLOAT32" } // Outputs raw math values, perfect for CNNs
  };
}
function evaluatePixel(sample) {
  if (sample.dataMask === 0) return [-2]; // -2 will represent 'No Data' padded areas
  let ndwi = (sample.B03 - sample.B08) / (sample.B03 + sample.B08);
  return [ndwi];
}
"""

# The rasters will be stored in the directory
# data/dataset_ndwi

os.makedirs("../data/dataset_ndwi", exist_ok=True)
catalog_url = "https://sh.dataspace.copernicus.eu/api/v1/catalog/1.0.0/search"
process_url = 'https://sh.dataspace.copernicus.eu/api/v1/process'

# Loop over the 9 reservoirs, and each of the
# years in the period 

for dam_name, info in catalan_dams.items():
    print(f"\nProcessing {dam_name}...")
    bbox = get_custom_bbox(info['lat'], info['lon'], info['width_km'], info['height_km'])
    
    # Calculate fixed 20m/pixel resolution
    pixels_x = int((info["width_km"] * 1000) / 20)
    pixels_y = int((info["height_km"] * 1000) / 20)
    
    # We loop year-by-year to avoid Catalog API pagination limits (max 100 results per call)
    for year in range(2016, 2027):
        start_date = f"{year}-01-01T00:00:00Z"
        end_date = f"{year}-12-31T23:59:59Z"
        if year == 2026:
            end_date = "2026-03-05T23:59:59Z"
            
        print(f"  Searching Catalog for {year}...")
        
        # Ask Catalog API for valid dates
        catalog_payload = {
            "collections": ["sentinel-2-l2a"],
            "datetime": f"{start_date}/{end_date}",
            "bbox": bbox,
            "filter": "eo:cloud_cover <= 15",
            "filter-lang": "cql2-text",
            "limit": 100
        }
        
        cat_res = requests.post(catalog_url, headers=catalog_headers, json=catalog_payload)
        
        # Warning if it fails
        if cat_res.status_code != 200:
            print(f"  Catalog API Error: {cat_res.text}")
            continue
        
        cat_res = requests.post(catalog_url, headers=catalog_headers, json=catalog_payload)
        features = cat_res.json().get('features', [])
        
        # Extract unique days (sometimes Sentinel takes 2 overlapping shots on the same day)
        valid_dates = sorted(list(set([f['properties']['datetime'][:10] for f in features])))
        
        print(f"  Found {len(valid_dates)} clear days in {year}. Downloading tensors...")
        
        # Download NDWI for those exact dates
        for date_str in valid_dates:
            filepath = f"../data/dataset_ndwi/{dam_name}_{date_str}.tiff"
            
            # Skip if we already downloaded it (useful if script crashes and we restart)
            if os.path.exists(filepath):
                continue 
                
            process_payload = {
                "input": {
                    "bounds": {"bbox": bbox},
                    "data": [{
                        "dataFilter": {
                            "timeRange": {
                                "from": f"{date_str}T00:00:00Z", 
                                "to": f"{date_str}T23:59:59Z"
                            }
                        },
                        "type": "sentinel-2-l2a"
                    }]
                },
                "output": {
                    "width": pixels_x, "height": pixels_y,
                    "responses": [{"identifier": "default", "format": {"type": "image/tiff"}}]
                },
                "evalscript": evalscript_ndwi
            }
            
            proc_res = requests.post(process_url, headers=headers, json=process_payload)
            
            if proc_res.status_code == 200:
                with open(filepath, 'wb') as f:
                    f.write(proc_res.content)
            else:
                print(f"    Failed to download {date_str}: {proc_res.status_code}")

print("\nDataset generation complete!")


Processing riudecanyes...
  Searching Catalog for 2016...
  Found 24 clear days in 2016. Downloading tensors...
  Searching Catalog for 2017...
  Found 59 clear days in 2017. Downloading tensors...
  Searching Catalog for 2018...
  Found 46 clear days in 2018. Downloading tensors...
  Searching Catalog for 2019...
  Found 60 clear days in 2019. Downloading tensors...
  Searching Catalog for 2020...
  Found 44 clear days in 2020. Downloading tensors...
  Searching Catalog for 2021...
  Found 38 clear days in 2021. Downloading tensors...
  Searching Catalog for 2022...
  Found 51 clear days in 2022. Downloading tensors...
  Searching Catalog for 2023...
  Found 54 clear days in 2023. Downloading tensors...
  Searching Catalog for 2024...
  Found 55 clear days in 2024. Downloading tensors...
  Searching Catalog for 2025...
  Found 52 clear days in 2025. Downloading tensors...
  Searching Catalog for 2026...
  Found 3 clear days in 2026. Downloading tensors...

Processing siurana...
  Sea